# Period 1: The menu database

Practise bench for **setup_db.py**, the **`/menu` route** and **menu.html**.

**How this notebook works**
1. **Practise here first.** Write and test each piece in the cells below. A cell that says **TODO** is yours to write.
2. Click a cell and press **Shift+Enter** to run it. Run the cells **in order, from the top**.
3. A check cell prints **✓** when you are right, or **✗** and a hint when you are not. Fix the cell above it and run both again.
4. When every check shows ✓, **copy your code into the real app files** (the last section tells you where).

If a cell shows a red error such as `no such table`, you skipped a TODO cell above it. Run that one first.

The database in this notebook is a **practice copy that lives in memory**. It is not your real `ais.db`, so you cannot break anything.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")   # a practice database that lives in memory
db.row_factory = sqlite3.Row       # so each row can be read like a dictionary: row["name"]

def show(sql, params=()):
    """Run a SELECT and print the rows as a small table. (Given: you do not need to read this.)"""
    cur = db.execute(sql, params)
    rows = cur.fetchall()
    names = [d[0] for d in cur.description]
    widths = [max([len(str(n))] + [len(str(r[n])) for r in rows]) for n in names]
    print("  ".join(str(n).ljust(w) for n, w in zip(names, widths)))
    print("  ".join("-" * w for w in widths))
    for r in rows:
        print("  ".join(str(r[n]).ljust(w) for n, w in zip(names, widths)))
    print("(" + str(len(rows)) + " rows)")

def check(ok, good, hint):
    """Print a tick or a cross. (Given.)"""
    if ok:
        print("✓ " + good)
    else:
        print("✗ " + hint)

## 1. Build the menu table

You write SQL to make a table and fill it with rows. Here is the shape (the Period 1 slides and guide show the same):

```sql
DROP TABLE IF EXISTS menu;
CREATE TABLE menu (
    id    INTEGER PRIMARY KEY AUTOINCREMENT,
    name  TEXT NOT NULL,
    price REAL NOT NULL
);
INSERT INTO menu (name, price) VALUES ('Nasi lemak', 3.50);
INSERT INTO menu (name, price) VALUES ('Singapore laksa', 4.00);
```

- Text goes in **single quotes**, numbers do not.
- Leave out `id`: it counts up by itself.
- Every statement ends with a **semicolon**.

### TODO 1: your menu SQL

Write the `DROP`, the `CREATE TABLE`, and **at least seven** `INSERT` statements for your own canteen items. Put them between the triple quotes.

In [ ]:
db.executescript("""
-- TODO 1: write your menu SQL here.
-- 1. DROP TABLE IF EXISTS menu;
-- 2. CREATE TABLE menu with the columns id, name, price
-- 3. at least seven INSERT statements, one per item
""")

In [ ]:
cols = [r["name"] for r in db.execute("PRAGMA table_info(menu)")]
check(cols == ["id", "name", "price"], "The menu table has the columns id, name, price.",
      "Expected the columns id, name, price. Found: " + str(cols) + ". Check your CREATE TABLE, then run the TODO 1 cell again.")

n = db.execute("SELECT COUNT(*) FROM menu").fetchone()[0] if cols else 0
check(n >= 7, "The menu table has " + str(n) + " rows.",
      "Only " + str(n) + " rows so far. Add INSERT statements until you have at least seven, then run the TODO 1 cell again.")

bad = [r["name"] for r in db.execute("SELECT name, price FROM menu")
       if not isinstance(r["price"], (int, float)) or r["price"] <= 0] if cols else []
check(len(bad) == 0 and n > 0, "Every item has a number for its price.",
      "These items do not have a proper price: " + str(bad) + ". Prices are numbers, so leave them out of quotes.")

## 2. Read the menu with SELECT

`SELECT` asks the database for rows. `show(...)` is a helper that prints them as a table.

In [ ]:
show("SELECT id, name, price FROM menu")

### TODO 2: a cheaper menu

Use `show(...)` with a `WHERE` and an `ORDER BY` to list only the items that cost **less than 4.00**, cheapest first.

In [ ]:
# TODO 2: write your show(...) line here.

## 3. Bring the rows into Python

In the app, `/menu` runs a `SELECT` and keeps the result with `fetchall()`. You get a **list of rows**, and each row is read by its column name, like a dictionary. This is the list of dictionaries from Period 0.

In [ ]:
rows = db.execute("SELECT id, name, price FROM menu").fetchall()

print(type(rows), len(rows))
first = rows[0]
print(first["name"], first["price"])

### TODO 3: print every item

Use a `for` loop over `rows` to print each item like this: `Nasi lemak costs 3.5`

In [ ]:
# TODO 3: write your for loop here.

## 4. Show every item on the page: the Jinja loop

A page is built from a **template**. Jinja is the template language: `{{ item.name }}` prints a value, and `{% for item in menu %} ... {% endfor %}` repeats the part between them once for every row.

The cell below builds a small page from `rows`. It does not show a loop yet.

### TODO 4: add the loop

Replace the words `(write your loop here)` with a Jinja loop that makes one `<li>` for every item. Each `<li>` shows the item's name and price.

In [ ]:
from jinja2 import Template

page = Template("""
<ul class="menu-list">
  (write your loop here)
</ul>
""")

html = page.render(menu=rows)
print(html)

In [ ]:
count = html.count("<li")
check(count == len(rows), "Your page has one <li> for each of the " + str(len(rows)) + " items.",
      "Your page has " + str(count) + " <li> items but there are " + str(len(rows)) + " rows. Put the <li> inside the for loop.")
missing = [r["name"] for r in rows if r["name"] not in html]
check(len(missing) == 0, "Every item's name is on the page.",
      "These names are missing from the page: " + str(missing) + ". Use {{ item.name }} inside the loop.")

## 5. Move it into the real app

When every check above shows ✓, copy your code across:

| What | From this notebook | Into |
| --- | --- | --- |
| The menu SQL | the TODO 1 cell, **only the SQL between the triple quotes** | `setup_db.py`, between the **first** pair of triple quotes |
| The query | `SELECT id, name, price FROM menu` | `main.py`, in the `/menu` route (replace the TODO and `items = []`) |
| The loop | your `{% for %}` ... `{% endfor %}` from TODO 4 | `templates/menu.html`, in place of the sample item |

Then, in the terminal, run `python3 setup_db.py`, and run the check cell below.

In [ ]:
import os, shutil, tempfile, importlib

def load_app():
    """Load your main.py. (Given: you do not need to read this.)"""
    import main
    importlib.reload(main)
    return main

def app_with_copy_of_db(main):
    """Point the app at a COPY of your ais.db, so these checks never change your real data."""
    tmp = os.path.join(tempfile.mkdtemp(), "check.db")
    if os.path.exists("ais.db"):
        shutil.copy("ais.db", tmp)
    main.DB_PATH = tmp
    main.init_db()          # makes the users table if it is missing
    return main

In [ ]:
def check_my_app():
    """Check your real files. (Given: you do not need to read this.)"""
    if not os.path.exists("ais.db"):
        check(False, "", "There is no ais.db yet. Put your SQL in setup_db.py and run python3 setup_db.py in the terminal.")
        return
    real = sqlite3.connect("ais.db")
    try:
        n = real.execute("SELECT COUNT(*) FROM menu").fetchone()[0]
    except sqlite3.Error:
        check(False, "", "ais.db has no menu table yet. Put your SQL in setup_db.py and run python3 setup_db.py.")
        return
    check(n >= 7, "ais.db has a menu table with " + str(n) + " rows.", "ais.db has only " + str(n) + " menu rows. Add more INSERT statements in setup_db.py and run it again.")
    main = load_app()
    r = main.app.test_client().get("/menu")
    page = r.get_data(as_text=True)
    buttons = page.count(">Order<")
    check(r.status_code == 200 and buttons == n,
          "/menu in your app shows all " + str(n) + " items, each with an Order button.",
          "/menu shows " + str(buttons) + " Order button(s) but your database has " + str(n) + " items. Check the SELECT in main.py and the loop in menu.html.")

check_my_app()